# Layers

> Potentially helpful layers for your models

In [ ]:
#| default_exp layers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, numpy as np, torch.nn.functional as F

from torch import nn
from physiojepa.augmentations import create_patch, mask_patches_simple, jitter_augmentation, shuffle_dim, reverse_sequence, channel_masking
from rotary_embedding_torch import RotaryEmbedding
from torch.nn.attention import SDPBackend, sdpa_kernel

## Miscellaneous

In [ ]:
#| export
class Identity(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x, **kwargs):
        return x
    
class Transpose(nn.Module):
    def __init__(self, *dims, contiguous=False): 
        super().__init__()
        self.dims, self.contiguous = dims, contiguous
    def forward(self, x):        
        if self.contiguous: return x.transpose(*self.dims).contiguous()
        else: return x.transpose(*self.dims)

def get_activation_fn(activation):
    if callable(activation): return activation()
    elif activation.lower() == "relu": return nn.ReLU()
    elif activation.lower() == "gelu": return nn.GELU()
    raise ValueError(f'{activation} is not available. You can use "relu", "gelu", or a callable')

## Positional Encoding Layers

In [ ]:
#| export
class PositionalEncoding(nn.Module):
    def __init__(self, 
                 num_patch, # number of patches of time series or stft in input
                 d_model, # dimension of patch embeddings
                 #dropout=0.1 # dropout value
                 ):
        super().__init__()
        self.num_patch = num_patch
        self.d_model = d_model
        
        # Positional encoding - learned
        self.W_pos =  nn.Parameter(torch.empty((num_patch, d_model)))
        nn.init.uniform_(self.W_pos, -0.02, 0.02)
        self.scale_factor = nn.Parameter(torch.ones(1))
        #self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        """
        input: x: [bs * nvars x num_patch x d_model]
        returns: x: [bs * nvars x num_patch x d_model]
        """
        x = x + self.scale_factor * self.W_pos
        return x

In [ ]:
#| export
class tAPE(nn.Module):
    """
    time Absolute Position Encoding
    Adapted from tsai
    """

    def __init__(self, 
        d_model:int, # the embedding dimension
        seq_len:int, # the max. length of the incoming sequence or num patches
        ):
        super().__init__()
        self.scale_factor = nn.Parameter(torch.ones(1)) # learnable scale factor
        W_pos = torch.zeros(seq_len, d_model)  # positional encoding
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        W_pos[:, 0::2] = torch.sin((position * div_term)*(d_model/seq_len)) # this is the difference between normal PE and tAPE, scaling (d_model/seq_len)
        W_pos[:, 1::2] = torch.cos((position * div_term)*(d_model/seq_len))
        self.register_buffer('W_pos', W_pos)  # this stores the variable in the state_dict (used for non-trainable variables)
        #self.W_pos = self.scale_factor * self.W_pos.unsqueeze(0)
        

        #self.dropout = nn.Dropout(p=dropout)

    def forward(self, x): # [batch size, sequence length, embed dim]
        x = x + self.W_pos.to(x.device) * self.scale_factor.to(x.device)
        return x
    


## Mask and Augmentation Layers

In [ ]:
#| export
class PatchAugmentations(nn.Module):
    def __init__(self, augmentations=['patch_mask', 'jitter_zero_mask', 'reverse_sequence', 'shuffle_channels', 'channel_masking'], patch_mask_ratio=0., jitter_zero_mask_ratio=0.):
        super().__init__()
        self.augmentations = augmentations
        self.patch_mask_ratio = patch_mask_ratio
        self.jitter_zero_mask_ratio = jitter_zero_mask_ratio 
    def forward(self, x):
        if 'patch_mask' in self.augmentations and 'jitter_zero_mask' in self.augmentations:
            c = ['patch_mask', 'jitter_zero_mask']
            mask_choice = c[torch.randperm(2)[0].item()] #np.random.choice(['patch_mask', 'jitter_zero_mask'], replace=True)
        else:
            mask_choice = None
        augs = self.augmentations.copy()
        augs = [augs[i] for i in torch.randperm(len(augs)).tolist()]
        for augmentation in augs:
            if augmentation == 'jitter_zero_mask' and (mask_choice == 'jitter_zero_mask' or mask_choice is None):
                if isinstance(self.jitter_zero_mask_ratio, tuple) or isinstance(self.jitter_zero_mask_ratio, list):
                    mask_ratio = self.jitter_zero_mask_ratio[0] + torch.rand(1).item() * (self.jitter_zero_mask_ratio[1] - self.jitter_zero_mask_ratio[0])
                else:
                    mask_ratio = self.jitter_zero_mask_ratio
                # mask is a number with the number of masks applied in this function
                x = jitter_augmentation(x, mask_ratio=mask_ratio, jitter_ratio=mask_ratio)
            if augmentation == 'patch_mask' and (mask_choice == 'patch_mask' or mask_choice is None):
                # padding mask is currently not implemented here.. tbh not sure if its needed
                if isinstance(self.patch_mask_ratio, tuple) or isinstance(self.patch_mask_ratio, list):
                    mask_ratio = self.patch_mask_ratio[0] + torch.rand(1).item() * (self.patch_mask_ratio[1] - self.patch_mask_ratio[0])
                else:
                    mask_ratio = self.patch_mask_ratio
                x = mask_patches_simple(x, mask_ratio=mask_ratio)
            if augmentation == 'channel_masking':
                x = channel_masking(x, dim=2, p=0.5)
            if augmentation == 'shuffle_channels':
                x = shuffle_dim(x, dim=2, p=0.5)
            if augmentation == 'reverse_sequence':
                x = reverse_sequence(x, seq_dim=(-1,), p=0.5)
        return x

## Patch and Fourier Layers

In [ ]:
#| export
class Patch(nn.Module):
    def __init__(self, patch_len, stride):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
    
    def forward(self, x, constant_pad=False, constant_pad_value=0):
        x = create_patch(x, patch_len=self.patch_len, stride=self.stride, constant_pad=constant_pad, constant_pad_value=constant_pad_value)
        return x

## Reversible Instance Normalization

In [ ]:
#| export
class RevIN(nn.Module):
    def __init__(self, 
                 num_features: int, # the number of channels or features in the input
                 eps=1e-5, # added to avoid division by zero errors
                 dim_to_reduce=-1, # the dimension to reduce, 
                 affine=True # learning affine parameters bias and weight per channel
                 ):
        """
        
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        self.dim_to_reduce = dim_to_reduce

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features,1))
            self.affine_bias = nn.Parameter(torch.zeros(num_features,1))

    def forward(self, x, mode:bool):
        """
        x: [bs x n_vars x max_seq_len]
        """
        if mode:
            return self._normalize(x)
        else:
            return self._denormalize(x)

    def _normalize(self, x):
        self.mean = torch.mean(x, dim=self.dim_to_reduce, keepdim=True).detach()
        self.stdev = torch.std(x, dim=self.dim_to_reduce, keepdim=True, unbiased=False).detach() + self.eps
        x = x.sub(self.mean)
        x = x.div(self.stdev)
        if self.affine:
            x = x.mul(self.affine_weight)
            x = x.add(self.affine_bias)
        return x

    def _denormalize(self, x):
        if self.affine:
            x = x.sub(self.affine_bias)
            x = x.div(self.affine_weight)
        x = x.mul(self.stdev)
        x = x.add(self.mean)
        return x

## Inception

In [ ]:
#| export
class InceptionModule(nn.Module):
    """
    Inception module adapted from https://github.com/timeseriesAI/tsai/blob/main/tsai/models/InceptionTime.py
    """
    def __init__(self, 
                 in_channels: int,
                 bottleneck_channels: int = 32,
                 bottleneck = True,
                 kernel_size: int = 40,
                 groups: int = 1
                 ):
        super().__init__()
        kernel_size = [kernel_size // (2**i) for i in range(3)]
        kernel_size = [k if k % 2 != 0 else k - 1 for k in kernel_size]  # ensure odd ks
        bottleneck = bottleneck if in_channels > 1 else False
        assert bottleneck_channels % groups == 0, f"bottleneck_channels ({bottleneck_channels}) must be divisible by groups ({groups})"
        assert in_channels % groups == 0, f"in_channels ({in_channels}) must be divisible by groups ({groups})"
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, 1, bias=False, groups=groups) if bottleneck else nn.Identity()
        self.convs = nn.ModuleList([nn.Conv1d(bottleneck_channels if bottleneck else in_channels, bottleneck_channels, k, padding=k//2, bias=False, groups=groups) for k in kernel_size])
        self.maxconvpool = nn.Sequential(*[nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(in_channels, bottleneck_channels, 1, bias=False, groups=groups)])
        self.bn = nn.BatchNorm1d(bottleneck_channels * 4)
        self.act = nn.ReLU()

    def forward(self, x):
        input_tensor = x
        x = self.bottleneck(input_tensor)
        x = torch.cat([l(x) for l in self.convs] + [self.maxconvpool(input_tensor)], dim=1)
        return self.act(self.bn(x))

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, bottleneck_channels=32, residual=True, depth=6, groups=1, **kwargs):
        super().__init__()
        self.residual, self.depth = residual, depth
        self.inception, self.shortcut = nn.ModuleList(), nn.ModuleList()
        self.bottleneck_channels = bottleneck_channels
        self.groups = groups
        for d in range(depth):
            self.inception.append(InceptionModule(in_channels if d == 0 else bottleneck_channels * 4, bottleneck_channels, groups=groups, **kwargs))
            if self.residual and d % 3 == 2: 
                n_in, n_out = in_channels if d == 2 else bottleneck_channels * 4, bottleneck_channels * 4
                if n_in == n_out:
                    self.shortcut.append(nn.BatchNorm1d(n_in))
                else:
                    self.shortcut.append(nn.Sequential(nn.Conv1d(n_in, n_out, 1, bias=False, groups=groups), nn.BatchNorm1d(n_out)))
        self.act = nn.ReLU()
        
    def forward(self, x):
        res = x
        for d, l in enumerate(range(self.depth)):
            x = self.inception[d](x)
            if self.residual and d % 3 == 2: 
                y = self.shortcut[d//3](res)
                x = x.add(y)
                res = x = self.act(x)
        return x

## Attention

In [ ]:
#| export
class MLP(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features=None,
        out_features=None,
        act_layer=nn.GELU,
        drop=0.
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=8,
        qkv_bias=False,
        qk_scale=None,
        attn_drop=0.,
        proj_drop=0.,
        rotary_pes=False
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = qk_scale or self.head_dim ** -0.5
        #self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.W_Q = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_K = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_V = nn.Linear(dim, dim, bias=qkv_bias)
        self.attn_drop_prob = attn_drop
        #self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.rotary_pes = rotary_pes
        if self.rotary_pes:
            self.rotary_embed = RotaryEmbedding(dim=self.head_dim,
                                                freqs_for="lang",
                                                theta=10000,
                                                learned_freq=False,
                                                seq_before_head_dim=False,
                                                use_xpos=False,
                                                cache_max_seq_len=29000 # max n patches ? 
                                                )

    def forward(self, x, key=None, value=None, mask=None):
        """
        Attention
        """
        if key is None:
            key = x
        if value is None:
            value = x
        # Split and reshape 
        q = self.W_Q(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        k = self.W_K(key).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        v = self.W_V(value).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        if self.rotary_pes:
            if not self.training:
                q, k = self.rotary_embed.rotate_queries_with_cached_keys(q, k)
            else:
                q = self.rotary_embed.rotate_queries_or_keys(q)
                k = self.rotary_embed.rotate_queries_or_keys(k)
        #B, N, C = x.shape
        #qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        #q, k, v = qkv[0], qkv[1], qkv[2]
        attn_dropout = self.attn_drop_prob if self.training else 0.0
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH], set_priority=True):
             x = F.scaled_dot_product_attention(q, k, v, dropout_p=attn_dropout, is_causal=False, scale=self.scale)
        # attn = (q @ k.transpose(-2, -1)) * self.scale  # [B, num_heads, D, D]
        # attn = attn.softmax(dim=-1)
        # attn = self.attn_drop(attn)
        #x = (attn @ v)

        x = x.transpose(1, 2).flatten(-2)
        #x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()